## Лабораторная работа №4.

Как вы знаете, во время написания художественного текста авторы нередко испытывают муки выбора между тем или иным словом. Все эти правки остаются за кадром: текстологи изучают черновики и моделируют творческий процесс, анализируют замены и пытаются отследить изменения в авторском замысле.

Черновики читать сложно! Но всем однажды хочется заглянуть в закулисье творческого процесса. Давайте напишем текст, которого никогда не было, но который вполне мог бы существовать!

Задача: возьмите отрывок любого художественного текста (абзац или несколько строф). Замените все 
- существительные, 
- глаголы,
- прилагательные

на ближайшие ассоциаты той же части речи из векторной модели. 


### Критерии оценивания:
1. Художественный текст токенизирован и лемматизирован. - 2 балла
2. Для знаменательных частей речи извлечены грамматические характеристики. - 2,5 балла
3. Для знаменательных частей речи подобран ближайший семантический ассоциат. - 2,5 балла
4. Замены всех частей речи стоят в правильной форме. - 3 балла

In [2]:
import re
import gensim
from razdel import tokenize
import pymorphy3

morph = pymorphy3.MorphAnalyzer()
model = gensim.models.KeyedVectors.load_word2vec_format(r"C:\Users\krist\Desktop\Учеба_Программирование\model.bin", binary=True)


text = """У лукоморья дуб зелёный;
Златая цепь на дубе том:
И днём и ночью кот учёный
Всё ходит по цепи кругом;
Идёт направо — песнь заводит,
Налево — сказку говорит."""

print("Исходный текст:")
print(text)

target_pos = {'NOUN', 'VERB', 'ADJF', 'ADJS', 'INFN'}

Исходный текст:
У лукоморья дуб зелёный;
Златая цепь на дубе том:
И днём и ночью кот учёный
Всё ходит по цепи кругом;
Идёт направо — песнь заводит,
Налево — сказку говорит.


In [12]:
tokens_info = [] 

for t in tokenize(text):
    word = t.text
    if not re.match(r'\w+', word, re.UNICODE):
        tokens_info.append({'text': word, 'is_word': False})
        continue

    parsed = morph.parse(word)[0]
    pos = parsed.tag.POS
    lemma = parsed.normal_form

    grammemes = {
        'pos': pos,
        'case': parsed.tag.case,
        'number': parsed.tag.number,
        'gender': parsed.tag.gender,
        'tense': parsed.tag.tense,
        'person': parsed.tag.person,
        'animacy': parsed.tag.animacy,
        'aspect': parsed.tag.aspect,
    }

    tokens_info.append({
        'text': word,
        'is_word': True,
        'lemma': lemma,
        'pos': pos,
        'grammemes': grammemes,
        'parsed': parsed
    })

print('Слово -> лемма, часть речи, грамм. характеристики')
for info in tokens_info:
    if info['is_word'] and info['pos'] in target_pos:
        g = info['grammemes']
        print(info['text'], '->', info['lemma'], ',', info['pos'],
              ', падеж:', g['case'], ', число:', g['number'],
              ', род:', g['gender'], ', время:', g['tense'])

Слово -> лемма, часть речи, грамм. характеристики
лукоморья -> лукоморье , NOUN , падеж: gent , число: sing , род: neut , время: None
дуб -> дуб , NOUN , падеж: nomn , число: sing , род: masc , время: None
зелёный -> зелёный , ADJF , падеж: nomn , число: sing , род: masc , время: None
Златая -> златой , ADJF , падеж: nomn , число: sing , род: femn , время: None
цепь -> цепь , NOUN , падеж: accs , число: sing , род: femn , время: None
дубе -> дуб , NOUN , падеж: loct , число: sing , род: masc , время: None
том -> тот , ADJF , падеж: loct , число: sing , род: neut , время: None
днём -> день , NOUN , падеж: ablt , число: sing , род: masc , время: None
кот -> кот , NOUN , падеж: nomn , число: sing , род: masc , время: None
учёный -> учёный , NOUN , падеж: nomn , число: sing , род: masc , время: None
ходит -> ходить , VERB , падеж: None , число: sing , род: None , время: pres
цепи -> цепь , NOUN , падеж: gent , число: sing , род: femn , время: None
Идёт -> идти , VERB , падеж: None , число:

In [13]:
def pos_for_model(pos):
    if pos == 'NOUN':
        return 'NOUN'
    if pos in ('VERB', 'INFN'):
        return 'VERB'
    if pos in ('ADJF', 'ADJS'):
        return 'ADJ'
    return None

In [14]:
def get_grammemes_for_inflect(parsed):
    
    gram = set()
    tag = parsed.tag
    for g in [tag.case, tag.number, tag.gender, tag.tense, tag.person, tag.animacy]:
        if g is not None:
            gram.add(g)
    return gram

In [15]:
for info in tokens_info:
    if not info['is_word']:
        continue
    pos = info['pos']
    if pos not in target_pos:
        continue

    mpos = pos_for_model(pos)
    key = info['lemma'] + '_' + mpos

    new_lemma = info['lemma']  
    try:
        similar = model.most_similar(key, topn=10)
        for cand, score in similar:
            if '_' in cand:
                cand_word, cand_pos = cand.rsplit('_', 1)
            else:
                cand_word, cand_pos = cand, ''
            if cand_pos == mpos and cand_word != info['lemma']:
                new_lemma = cand_word
                break
    except Exception as e:
        new_lemma = info['lemma']

    info['new_lemma'] = new_lemma

    gram = get_grammemes_for_inflect(info['parsed'])
    new_parsed = morph.parse(new_lemma)[0]
    new_form = None
    if gram:
        try:
            new_form = new_parsed.inflect(gram)
        except:
            new_form = None
    if new_form:
        new_word = new_form.word
    else:
        new_word = new_lemma

    if info['text'][0].isupper():
        new_word = new_word.capitalize()

    info['new_word'] = new_word

print('Слово - замена (лемма)')
for info in tokens_info:
    if info.get('new_word'):
        print(info['text'], '-', info['new_word'], '(', info.get('new_lemma'), ')')

Слово - замена (лемма)
лукоморья - лукоморья ( лукоморье )
дуб - дуба ( дуба )
зелёный - зелёный ( зеленый )
Златая - Злата ( златой )
цепь - цепочку ( цепочка )
дубе - дубе ( дуба )
том - кротком ( кротка )
днём - неделя ( неделя )
кот - кошка ( кошка )
учёный - физикохимик ( физикохимик )
ходит - водит ( водить )
цепи - цепочки ( цепочка )
Идёт - Пойти ( пойти )
песнь - трек ( трек )
заводит - заведёт ( завести )
сказку - сказку ( сказки )
говорит - сказать ( сказать )


In [16]:
result = []
for i, info in enumerate(tokens_info):
    if info['is_word'] and 'new_word' in info:
        result.append(info['new_word'])
    else:
        result.append(info['text'])

new_text = text
out = []
last = 0
for t, info in zip(tokenize(text), tokens_info):
    out.append(text[last:t.start]) 
    if info['is_word'] and 'new_word' in info:
        out.append(info['new_word'])
    else:
        out.append(info['text'])
    last = t.stop
out.append(text[last:])
new_text = ''.join(out)

print("Альтернативная версия текста:")
print(new_text)

Альтернативная версия текста:
У лукоморья дуба зелёный;
Злата цепочку на дубе кротком:
И неделя и ночью кошка физикохимик
Всё водит по цепочки кругом;
Пойти направо — трек заведёт,
Налево — сказку сказать.
